# Feature-Aligned Tuning with Dynamic-K

Evaluation includes both fixed `Top-11` and `dynamic-k` metrics under user-grouped nested CV.

In [13]:
from pathlib import Path
import warnings
import numpy as np
import pandas as pd

from sklearn.cluster import KMeans
from sklearn.model_selection import GroupKFold, RandomizedSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score, log_loss, precision_score, recall_score, f1_score

warnings.filterwarnings("ignore")

try:
    import lightgbm as lgb
    HAS_LIGHTGBM = True
except Exception:
    HAS_LIGHTGBM = False

print("HAS_LIGHTGBM:", HAS_LIGHTGBM)

HAS_LIGHTGBM: True


In [14]:
# ----------------------------
# Config
# ----------------------------
DATA_DIR = Path("data")
RANDOM_STATE = 42

# For speed while iterating. Set None for full users.
SAMPLE_USERS = 20000

OUTER_SPLITS = 3
INNER_SPLITS = 3
TOP_K_FIXED = 11
N_ITER_SEARCH = 12

print({
    "DATA_DIR": str(DATA_DIR),
    "SAMPLE_USERS": SAMPLE_USERS,
    "OUTER_SPLITS": OUTER_SPLITS,
    "INNER_SPLITS": INNER_SPLITS,
    "TOP_K_FIXED": TOP_K_FIXED,
    "N_ITER_SEARCH": N_ITER_SEARCH,
})

orders = pd.read_csv(DATA_DIR / "orders.csv")
op_prior = pd.read_csv(DATA_DIR / "order_products__prior.csv")
op_train = pd.read_csv(DATA_DIR / "order_products__train.csv")
products = pd.read_csv(DATA_DIR / "products.csv")[["product_id", "aisle_id"]]
aisles = pd.read_csv(DATA_DIR / "aisles.csv")[["aisle_id", "aisle"]]

print("orders:", orders.shape)
print("op_prior:", op_prior.shape)
print("op_train:", op_train.shape)
print("products:", products.shape)
print("aisles:", aisles.shape)

{'DATA_DIR': 'data', 'SAMPLE_USERS': 20000, 'OUTER_SPLITS': 3, 'INNER_SPLITS': 3, 'TOP_K_FIXED': 11, 'N_ITER_SEARCH': 12}
orders: (3421083, 7)
op_prior: (32434489, 4)
op_train: (1384617, 4)
products: (49688, 2)
aisles: (134, 2)


In [15]:
# ----------------------------
# Build candidate table and aligned features
# ----------------------------

# Target orders (the labeled final order for users)
target_orders = orders.loc[orders["eval_set"] == "train", ["order_id", "user_id", "order_number"]].copy()
target_orders = target_orders.rename(columns={"order_id": "target_order_id", "order_number": "target_order_number"})

if SAMPLE_USERS is not None:
    rng = np.random.default_rng(RANDOM_STATE)
    sample_uids = rng.choice(target_orders["user_id"].unique(), size=min(SAMPLE_USERS, target_orders["user_id"].nunique()), replace=False)
    target_orders = target_orders[target_orders["user_id"].isin(sample_uids)].copy()

# Prior orders for selected users (attach temporal fields)
prior_orders = orders.loc[orders["eval_set"] == "prior", [
    "order_id", "user_id", "order_number", "days_since_prior_order", "order_dow", "order_hour_of_day"
]].copy()
prior_orders = prior_orders.merge(
    target_orders[["user_id", "target_order_id", "target_order_number"]],
    on="user_id",
    how="inner",
)

# History purchase rows
history = op_prior.merge(
    prior_orders[["order_id", "user_id", "order_number", "days_since_prior_order", "target_order_number"]],
    on="order_id",
    how="inner",
)

# Positive labels from target order
positives = op_train.merge(
    target_orders[["target_order_id", "user_id"]],
    left_on="order_id",
    right_on="target_order_id",
    how="inner",
)
positives = positives[["user_id", "product_id"]].drop_duplicates().assign(label=1)

# Candidate set: user historical products
candidates = history[["user_id", "product_id"]].drop_duplicates().copy()

# User-product aggregates
up_agg = (
    history.groupby(["user_id", "product_id"]).agg(
        up_buy_cnt=("order_id", "size"),
        up_reorder_cnt=("reordered", "sum"),
        up_reorder_ratio=("reordered", "mean"),
        up_last_order=("order_number", "max"),
        up_first_order=("order_number", "min"),
        up_avg_cart_order=("add_to_cart_order", "mean"),
        up_days_since_last_raw=("days_since_prior_order", "mean"),
    )
    .reset_index()
)

# Product aggregates
prod_agg = (
    history.groupby("product_id").agg(
        p_total_purchases=("order_id", "size"),
        p_reorder_ratio=("reordered", "mean"),
        p_avg_cart_order=("add_to_cart_order", "mean"),
        p_unique_users=("user_id", "nunique"),
    )
    .reset_index()
)

# User aggregates
user_order_size = history.groupby(["user_id", "order_id"]).size().rename("basket_size").reset_index()
user_agg = (
    history.groupby("user_id").agg(
        u_total_orders=("order_id", "nunique"),
        u_reorder_ratio=("reordered", "mean"),
        u_unique_products=("product_id", "nunique"),
        u_total_items=("order_id", "size"),
        u_avg_days_between_orders=("days_since_prior_order", "mean"),
    )
    .reset_index()
)
user_agg = user_agg.merge(
    user_order_size.groupby("user_id")["basket_size"].mean().rename("u_avg_basket_size").reset_index(),
    on="user_id",
    how="left",
)

# Merge base modeling table
model_df = candidates.merge(up_agg, on=["user_id", "product_id"], how="left")
model_df = model_df.merge(target_orders[["user_id", "target_order_number"]], on="user_id", how="left")
model_df = model_df.merge(prod_agg, on="product_id", how="left")
model_df = model_df.merge(user_agg, on="user_id", how="left")
model_df = model_df.merge(positives, on=["user_id", "product_id"], how="left")

model_df["label"] = model_df["label"].fillna(0).astype(int)

# Align engineered columns to project naming
model_df["up_orders_since_last"] = (model_df["target_order_number"] - model_df["up_last_order"]).clip(lower=0)
model_df["up_days_since_last"] = (model_df["up_orders_since_last"] * model_df["u_avg_days_between_orders"]).fillna(0)
model_df["up_freq"] = (model_df["up_buy_cnt"] / model_df["u_total_orders"].replace(0, np.nan)).fillna(0)

print("Users:", model_df["user_id"].nunique())
print("Rows:", model_df.shape[0])
print("Positive rate:", round(model_df["label"].mean(), 4))

Users: 20000
Rows: 1294496
Positive rate: 0.0973


In [18]:
# ----------------------------
# Apriori-derived feature: apriori_rule_hits
# Aligned with project description:
# - Build aisle-level rules from historical baskets
# - For each user, use aisles in the most recent historical basket
# - Mark hit if candidate aisle is a rule consequent of any recent-basket aisle
# ----------------------------

prod_aisle = products.merge(aisles, on="aisle_id", how="left")[["product_id", "aisle"]].copy()

history_with_aisle = history.merge(prod_aisle, on="product_id", how="left")

basket_aisles = (
    history_with_aisle.groupby("order_id")["aisle"]
    .apply(lambda x: sorted(set(x.dropna().astype(str))))
    .reset_index(name="aisle_basket")
)
basket_aisles = basket_aisles[basket_aisles["aisle_basket"].str.len() >= 2].copy()

single_counts = {}
pair_counts = {}
for basket in basket_aisles["aisle_basket"]:
    for item in basket:
        single_counts[item] = single_counts.get(item, 0) + 1
    for i in range(len(basket)):
        for j in range(i + 1, len(basket)):
            pair = (basket[i], basket[j])
            pair_counts[pair] = pair_counts.get(pair, 0) + 1


def build_aisle_rules(single_counts, pair_counts, n_baskets, min_support, min_confidence, min_lift):
    rows_local = []
    for (a, b), pair_cnt in pair_counts.items():
        support = pair_cnt / n_baskets
        conf_a_to_b = pair_cnt / single_counts[a]
        conf_b_to_a = pair_cnt / single_counts[b]
        lift_a_to_b = conf_a_to_b / (single_counts[b] / n_baskets)
        lift_b_to_a = conf_b_to_a / (single_counts[a] / n_baskets)

        rows_local.append({"antecedent": a, "consequent": b, "pair_count": pair_cnt, "support": support, "confidence": conf_a_to_b, "lift": lift_a_to_b})
        rows_local.append({"antecedent": b, "consequent": a, "pair_count": pair_cnt, "support": support, "confidence": conf_b_to_a, "lift": lift_b_to_a})

    rules_local = pd.DataFrame(rows_local)
    if len(rules_local) == 0:
        return rules_local

    rules_local = rules_local[
        (rules_local["support"] >= min_support)
        & (rules_local["confidence"] >= min_confidence)
        & (rules_local["lift"] >= min_lift)
    ].copy()

    if len(rules_local) == 0:
        return rules_local

    rules_local = rules_local.sort_values(["lift", "confidence", "pair_count"], ascending=[False, False, False]).reset_index(drop=True)
    return rules_local


rules_filtered = build_aisle_rules(
    single_counts=single_counts,
    pair_counts=pair_counts,
    n_baskets=max(len(basket_aisles), 1),
    min_support=0.01,
    min_confidence=0.30,
    min_lift=1.30,
)

ante_to_cons = {}
if len(rules_filtered) > 0:
    for a, b in rules_filtered[["antecedent", "consequent"]].itertuples(index=False):
        ante_to_cons.setdefault(a, set()).add(b)

# Most recent historical order per user
user_last_order = prior_orders.groupby("user_id")["order_number"].max().rename("last_order_number").reset_index()
last_orders = prior_orders.merge(user_last_order, on="user_id", how="inner")
last_orders = last_orders[last_orders["order_number"] == last_orders["last_order_number"]][["user_id", "order_id"]].drop_duplicates()

last_basket_aisles = (
    last_orders.merge(history_with_aisle[["order_id", "aisle"]], on="order_id", how="left")
    .groupby("user_id")["aisle"]
    .apply(lambda x: set(x.dropna().astype(str)))
)

# For each user, all consequent aisles triggered by their last basket aisles
user_allowed_aisles = {}
for uid, aisle_set in last_basket_aisles.items():
    allowed = set()
    for a in aisle_set:
        allowed.update(ante_to_cons.get(a, set()))
    user_allowed_aisles[uid] = allowed

model_df = model_df.merge(prod_aisle, on="product_id", how="left")
model_df["apriori_rule_hits"] = model_df.apply(
    lambda r: int(r["aisle"] in user_allowed_aisles.get(r["user_id"], set())),
    axis=1,
)
model_df = model_df.drop(columns=["aisle"])

print("Filtered rule count:", len(rules_filtered))
print("apriori_rule_hits mean:", round(model_df["apriori_rule_hits"].mean(), 4))

Filtered rule count: 148
apriori_rule_hits mean: 0.2148


In [21]:
# ----------------------------
# User clusters
# ----------------------------
cluster_source = user_agg[[
    "user_id",
    "u_total_orders",
    "u_avg_days_between_orders",
    "u_avg_basket_size",
    "u_reorder_ratio",
    "u_unique_products",
]].copy()

cluster_feature_cols = [
    "u_total_orders",
    "u_avg_days_between_orders",
    "u_avg_basket_size",
    "u_reorder_ratio",
    "u_unique_products",
]

for c in ["u_total_orders", "u_avg_basket_size", "u_unique_products"]:
    cluster_source[c] = np.log1p(cluster_source[c])

cluster_X = cluster_source[cluster_feature_cols].fillna(0)
cluster_scaler = StandardScaler()
cluster_X_scaled = cluster_scaler.fit_transform(cluster_X)

kmeans_user = KMeans(n_clusters=4, random_state=RANDOM_STATE, n_init=20)
cluster_source["user_cluster"] = kmeans_user.fit_predict(cluster_X_scaled)

# Make this cell rerunnable: remove previously created cluster columns before merge
cluster_cols_to_reset = [
    "user_cluster",
    "user_cluster_1",
    "user_cluster_2",
    "user_cluster_3",
    "cluster_product_target_rate",
]
model_df = model_df.drop(columns=[c for c in cluster_cols_to_reset if c in model_df.columns], errors="ignore")

model_df = model_df.merge(cluster_source[["user_id", "user_cluster"]], on="user_id", how="left")
model_df["user_cluster"] = model_df["user_cluster"].fillna(0).astype(int)

# user_cluster_1/2/3 (drop cluster_0 as baseline)
cluster_dummies = pd.get_dummies(model_df["user_cluster"], prefix="user_cluster", dtype=int)
for c in ["user_cluster_1", "user_cluster_2", "user_cluster_3"]:
    model_df[c] = cluster_dummies[c] if c in cluster_dummies.columns else 0

# cluster_product_target_rate from current sample
cp_rate = (
    model_df.groupby(["user_cluster", "product_id"])["label"]
    .mean()
    .rename("cluster_product_target_rate")
    .reset_index()
)
model_df = model_df.merge(cp_rate, on=["user_cluster", "product_id"], how="left")
model_df["cluster_product_target_rate"] = model_df["cluster_product_target_rate"].fillna(model_df["label"].mean())

print(cluster_source["user_cluster"].value_counts().sort_index())
print(model_df[["user_cluster", "cluster_product_target_rate", "user_cluster_1", "user_cluster_2", "user_cluster_3"]].head())

user_cluster
0    5176
1    3584
2    4684
3    6556
Name: count, dtype: int64
   user_cluster  cluster_product_target_rate  user_cluster_1  user_cluster_2  \
0             0                     0.030928               0               0   
1             0                     0.000000               0               0   
2             0                     0.046512               0               0   
3             0                     0.068182               0               0   
4             0                     0.062500               0               0   

   user_cluster_3  
0               0  
1               0  
2               0  
3               0  
4               0  


In [17]:
aligned_feature_cols = [
    "up_orders_since_last",
    "up_days_since_last",
    "up_freq",
    "up_buy_cnt",
    "up_reorder_ratio",
    "p_reorder_ratio",
    "u_total_orders",
    "cluster_product_target_rate",
    "p_avg_cart_order",
    "u_reorder_ratio",
    "u_unique_products",
    "p_total_purchases",
    "up_first_order",
    "u_avg_days_between_orders",
    "p_unique_users",
    "up_last_order",
    "u_total_items",
    "u_avg_basket_size",
    "up_avg_cart_order",
    "apriori_rule_hits",
    "user_cluster_1",
    "user_cluster_2",
    "user_cluster_3",
]

# Ensure all aligned columns exist
for c in aligned_feature_cols:
    if c not in model_df.columns:
        model_df[c] = 0

model_df[aligned_feature_cols] = model_df[aligned_feature_cols].replace([np.inf, -np.inf], np.nan).fillna(0)

id_cols = ["user_id", "product_id", "label"]
print("Model df:", model_df.shape)
print("Positive rate:", model_df["label"].mean())

Model df: (1294496, 29)
Positive rate: 0.09734058660667935


In [ ]:
def eval_row_level(y_true, y_prob):
    eps = 1e-15
    y_prob_clip = np.clip(y_prob, eps, 1 - eps)
    out = {
        "auc": roc_auc_score(y_true, y_prob) if len(np.unique(y_true)) > 1 else np.nan,
        "pr_auc": average_precision_score(y_true, y_prob),
        "logloss": log_loss(y_true, y_prob_clip),
    }
    return out


def eval_order_topk(df_eval, prob_col="prob", k=11, label_col="label"):
    ranked = df_eval.sort_values(["user_id", prob_col, "product_id"], ascending=[True, False, True]).copy()
    ranked["rank_within_user"] = ranked.groupby("user_id").cumcount() + 1
    ranked["pred_topk"] = (ranked["rank_within_user"] <= k).astype(int)

    order_precision, order_recall, order_f1, order_hit = [], [], [], []
    for _, g in ranked.groupby("user_id"):
        y_t = g[label_col].to_numpy()
        y_p = g["pred_topk"].to_numpy()
        order_precision.append(precision_score(y_t, y_p, zero_division=0))
        order_recall.append(recall_score(y_t, y_p, zero_division=0))
        order_f1.append(f1_score(y_t, y_p, zero_division=0))
        order_hit.append(float((g.loc[g["pred_topk"] == 1, label_col].sum() > 0)))

    return {
        f"precision@{k}": float(np.mean(order_precision)),
        f"recall@{k}": float(np.mean(order_recall)),
        f"f1@{k}": float(np.mean(order_f1)),
        f"hit@{k}": float(np.mean(order_hit)),
    }

# pred_k = round(u_avg_basket_size), clipped at >=1
def make_dynamic_k_map(df_input):
    user_k_df = df_input.groupby("user_id")["u_avg_basket_size"].first().reset_index()
    user_k_df["pred_k"] = np.rint(user_k_df["u_avg_basket_size"]).astype(int).clip(lower=1)
    return dict(zip(user_k_df["user_id"], user_k_df["pred_k"]))


def eval_order_dynamic_k(df_eval, prob_col, k_pred_map, label_col="label"):
    ranked = df_eval.sort_values(["user_id", prob_col, "product_id"], ascending=[True, False, True]).copy()
    ranked["rank_within_user"] = ranked.groupby("user_id").cumcount() + 1
    ranked["pred_k"] = ranked["user_id"].map(k_pred_map).fillna(TOP_K_FIXED).astype(int)
    ranked["pred_k"] = ranked["pred_k"].clip(lower=1)
    ranked["pred_topk_dynamic"] = (ranked["rank_within_user"] <= ranked["pred_k"]).astype(int)

    order_precision, order_recall, order_f1, order_hit, avg_pred_k = [], [], [], [], []
    for _, g in ranked.groupby("user_id"):
        y_t = g[label_col].to_numpy()
        y_p = g["pred_topk_dynamic"].to_numpy()
        order_precision.append(precision_score(y_t, y_p, zero_division=0))
        order_recall.append(recall_score(y_t, y_p, zero_division=0))
        order_f1.append(f1_score(y_t, y_p, zero_division=0))
        order_hit.append(float((g.loc[g["pred_topk_dynamic"] == 1, label_col].sum() > 0)))
        avg_pred_k.append(float(g["pred_k"].iloc[0]))

    return {
        "precision@dynamic_k": float(np.mean(order_precision)),
        "recall@dynamic_k": float(np.mean(order_recall)),
        "f1@dynamic_k": float(np.mean(order_f1)),
        "hit@dynamic_k": float(np.mean(order_hit)),
        "avg_pred_k": float(np.mean(avg_pred_k)),
    }

In [ ]:
X = model_df[aligned_feature_cols].copy()
y = model_df["label"].astype(int).to_numpy()
groups = model_df["user_id"].to_numpy()

outer_cv = GroupKFold(n_splits=OUTER_SPLITS)
inner_cv = GroupKFold(n_splits=INNER_SPLITS)

logreg_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(
        solver="saga",
        class_weight="balanced",
        random_state=RANDOM_STATE,
        n_jobs=-1,
        verbose=0,
    )),
])

logreg_param_dist = {
    "model__C": [0.03, 0.1, 0.3, 1.0, 3.0],
    "model__max_iter": [150, 250, 400, 600],
}

if HAS_LIGHTGBM:
    lgb_model = lgb.LGBMClassifier(
        objective="binary",
        random_state=RANDOM_STATE,
        n_jobs=-1,
        verbosity=-1,
    )
    lgb_param_dist = {
        "num_leaves": [31, 63, 127],
        "learning_rate": [0.02, 0.03, 0.05, 0.1],
        "n_estimators": [250, 400, 600],
        "min_child_samples": [20, 30, 50, 100],
        "subsample": [0.8, 1.0],
        "colsample_bytree": [0.8, 1.0],
    }
else:
    raise RuntimeError("LightGBM is required for this notebook. Please install lightgbm first.")

model_spaces = {
    "LogisticRegression_SAGA": (logreg_pipe, logreg_param_dist),
    "LightGBM": (lgb_model, lgb_param_dist),
}

print("Rows:", X.shape[0], "Features:", X.shape[1], "Users:", len(np.unique(groups)))

Rows: 1294496 Features: 23 Users: 20000


In [ ]:
rows = []

for model_name, (estimator, param_dist) in model_spaces.items():
    print(f"\n===== Tuning {model_name} =====")

    for fold_id, (tr_idx, te_idx) in enumerate(outer_cv.split(X, y, groups=groups), start=1):
        X_tr, X_te = X.iloc[tr_idx], X.iloc[te_idx]
        y_tr, y_te = y[tr_idx], y[te_idx]
        g_tr, g_te = groups[tr_idx], groups[te_idx]

        search = RandomizedSearchCV(
            estimator=estimator,
            param_distributions=param_dist,
            n_iter=N_ITER_SEARCH,
            scoring="roc_auc",
            cv=inner_cv,
            random_state=RANDOM_STATE,
            n_jobs=-1,
            refit=True,
            verbose=0,
        )
        search.fit(X_tr, y_tr, groups=g_tr)

        best_est = search.best_estimator_
        y_prob = best_est.predict_proba(X_te)[:, 1]

        fold_df = model_df.iloc[te_idx][["user_id", "product_id", "label", "u_avg_basket_size"]].copy()
        fold_df["prob"] = y_prob

        k_map = make_dynamic_k_map(fold_df)

        row = {
            "model": model_name,
            "outer_fold": fold_id,
            "inner_best_score_auc": float(search.best_score_),
            "best_params": str(search.best_params_),
        }
        row.update(eval_row_level(y_te, y_prob))
        row.update(eval_order_topk(fold_df, prob_col="prob", k=TOP_K_FIXED, label_col="label"))
        row.update(eval_order_dynamic_k(fold_df, prob_col="prob", k_pred_map=k_map, label_col="label"))

        rows.append(row)
        print(
            f"Fold {fold_id} | auc={row['auc']:.4f} | pr_auc={row['pr_auc']:.4f} | "
            f"f1@11={row['f1@11']:.4f} | f1@dynamic_k={row['f1@dynamic_k']:.4f}"
        )

results_df = pd.DataFrame(rows)
results_df


===== Tuning LogisticRegression_SAGA =====
Fold 1 | auc=0.8593 | pr_auc=0.4560 | f1@11=0.3615 | f1@dynamic_k=0.3971
Fold 2 | auc=0.8588 | pr_auc=0.4587 | f1@11=0.3646 | f1@dynamic_k=0.4006
Fold 3 | auc=0.8581 | pr_auc=0.4579 | f1@11=0.3659 | f1@dynamic_k=0.3995

===== Tuning LightGBM =====
Fold 1 | auc=0.8740 | pr_auc=0.4936 | f1@11=0.3715 | f1@dynamic_k=0.4116
Fold 2 | auc=0.8731 | pr_auc=0.4919 | f1@11=0.3750 | f1@dynamic_k=0.4155
Fold 3 | auc=0.8735 | pr_auc=0.4949 | f1@11=0.3748 | f1@dynamic_k=0.4143


,model,outer_fold,inner_best_score_auc,best_params,auc,pr_auc,logloss,precision@11,recall@11,f1@11,hit@11,precision@dynamic_k,recall@dynamic_k,f1@dynamic_k,hit@dynamic_k,avg_pred_k
0,LogisticRegression_SAGA,1,0.858389,"{'model__max_iter': 150, 'model__C': 0.03}",0.859335,0.455998,0.472595,0.311792,0.604910,0.361452,0.893205,0.350435,0.554069,0.397077,0.858257,9.987851
1,LogisticRegression_SAGA,2,0.858714,"{'model__max_iter': 150, 'model__C': 0.03}",0.858770,0.458707,0.470482,0.315842,0.608519,0.364555,0.896955,0.354458,0.559255,0.400634,0.868907,9.963702
2,LogisticRegression_SAGA,3,0.858978,"{'model__max_iter': 150, 'model__C': 0.03}",0.858109,0.457863,0.477053,0.314427,0.614699,0.365928,0.896640,0.353608,0.561310,0.399550,0.866637,9.980048
3,LightGBM,1,0.873146,"{'subsample': 0.8, 'num_leaves': 31, 'n_estima...",0.874015,0.493630,0.220798,0.321582,0.618252,0.371550,0.894105,0.363594,0.573860,0.411598,0.867107,9.987851
4,LightGBM,2,0.873478,"{'subsample': 0.8, 'num_leaves': 31, 'n_estima...",0.873091,0.491918,0.224051,0.326014,0.623156,0.374981,0.900855,0.368151,0.578392,0.415482,0.874756,9.963702
5,LightGBM,3,0.873366,"{'subsample': 0.8, 'num_leaves': 63, 'n_estima...",0.873476,0.494852,0.220343,0.322855,0.627104,0.374810,0.898590,0.367441,0.580539,0.414268,0.871737,9.980048


In [ ]:
summary_df = (
    results_df.groupby("model", as_index=False)
    .agg(
        auc_mean=("auc", "mean"),
        auc_std=("auc", "std"),
        pr_auc_mean=("pr_auc", "mean"),
        logloss_mean=("logloss", "mean"),
        p11_mean=("precision@11", "mean"),
        r11_mean=("recall@11", "mean"),
        f11_mean=("f1@11", "mean"),
        hit11_mean=("hit@11", "mean"),
        pdk_mean=("precision@dynamic_k", "mean"),
        rdk_mean=("recall@dynamic_k", "mean"),
        fdk_mean=("f1@dynamic_k", "mean"),
        hitdk_mean=("hit@dynamic_k", "mean"),
        avg_pred_k=("avg_pred_k", "mean"),
    )
    .sort_values("auc_mean", ascending=False)
)

print("=== Summary across outer folds ===")
summary_df

=== Summary across outer folds ===


,model,auc_mean,auc_std,pr_auc_mean,logloss_mean,p11_mean,r11_mean,f11_mean,hit11_mean,pdk_mean,rdk_mean,fdk_mean,hitdk_mean,avg_pred_k
0,LightGBM,0.873527,0.000464,0.493467,0.221731,0.323484,0.622837,0.373780,0.89785,0.366395,0.577597,0.413783,0.8712,9.9772
1,LogisticRegression_SAGA,0.858738,0.000614,0.457523,0.473376,0.314020,0.609376,0.363978,0.89560,0.352834,0.558211,0.399087,0.8646,9.9772


In [ ]:
# ----------------------------
# MLP tuning
# Rationale:
# - LightGBM > LogReg by a clear margin, so MLP should be regularized and stable first.
# ----------------------------

from sklearn.neural_network import MLPClassifier

mlp_pipe = Pipeline([
    ("scaler", StandardScaler()),
    (
        "model",
        MLPClassifier(
            random_state=RANDOM_STATE,
            early_stopping=True,
            validation_fraction=0.1,
            n_iter_no_change=8,
            max_iter=80,
            verbose=False,
        ),
    ),
])

mlp_param_dist = {
    # Moderate-size nets to control runtime and overfit risk
    "model__hidden_layer_sizes": [(64,), (128,), (128, 64)],
    # Stronger regularization range due to class imbalance/noisy negatives
    "model__alpha": [1e-4, 3e-4, 1e-3, 3e-3],
    # Conservative learning rates for stable convergence
    "model__learning_rate_init": [3e-4, 6e-4, 1e-3],
    "model__batch_size": [256, 512, 1024],
}

mlp_rows = []
print("\n===== Tuning MLP =====")

for fold_id, (tr_idx, te_idx) in enumerate(outer_cv.split(X, y, groups=groups), start=1):
    X_tr, X_te = X.iloc[tr_idx], X.iloc[te_idx]
    y_tr, y_te = y[tr_idx], y[te_idx]
    g_tr, g_te = groups[tr_idx], groups[te_idx]

    mlp_search = RandomizedSearchCV(
        estimator=mlp_pipe,
        param_distributions=mlp_param_dist,
        n_iter=10,
        scoring="roc_auc",
        cv=inner_cv,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        refit=True,
        verbose=0,
    )
    mlp_search.fit(X_tr, y_tr, groups=g_tr)

    mlp_best = mlp_search.best_estimator_
    mlp_prob = mlp_best.predict_proba(X_te)[:, 1]

    fold_df = model_df.iloc[te_idx][["user_id", "product_id", "label", "u_avg_basket_size"]].copy()
    fold_df["prob"] = mlp_prob
    k_map = make_dynamic_k_map(fold_df)

    row = {
        "model": "MLP",
        "outer_fold": fold_id,
        "inner_best_score_auc": float(mlp_search.best_score_),
        "best_params": str(mlp_search.best_params_),
    }
    row.update(eval_row_level(y_te, mlp_prob))
    row.update(eval_order_topk(fold_df, prob_col="prob", k=TOP_K_FIXED, label_col="label"))
    row.update(eval_order_dynamic_k(fold_df, prob_col="prob", k_pred_map=k_map, label_col="label"))

    mlp_rows.append(row)
    print(
        f"Fold {fold_id} | auc={row['auc']:.4f} | pr_auc={row['pr_auc']:.4f} | "
        f"f1@11={row['f1@11']:.4f} | f1@dynamic_k={row['f1@dynamic_k']:.4f}"
    )

mlp_results_df = pd.DataFrame(mlp_rows)
mlp_results_df


===== Tuning MLP =====
Fold 1 | auc=0.8751 | pr_auc=0.4940 | f1@11=0.3724 | f1@dynamic_k=0.4125
Fold 2 | auc=0.8733 | pr_auc=0.4936 | f1@11=0.3748 | f1@dynamic_k=0.4151
Fold 3 | auc=0.8736 | pr_auc=0.4927 | f1@11=0.3759 | f1@dynamic_k=0.4146


,model,outer_fold,inner_best_score_auc,best_params,auc,pr_auc,logloss,precision@11,recall@11,f1@11,hit@11,precision@dynamic_k,recall@dynamic_k,f1@dynamic_k,hit@dynamic_k,avg_pred_k
0,MLP,1,0.872876,"{'model__learning_rate_init': 0.001, 'model__h...",0.875062,0.494036,0.221031,0.322237,0.619823,0.372426,0.895305,0.364192,0.575417,0.412489,0.868757,9.987851
1,MLP,2,0.873803,"{'model__learning_rate_init': 0.001, 'model__h...",0.873251,0.493640,0.224219,0.325837,0.622230,0.374764,0.900105,0.368049,0.577871,0.415148,0.872956,9.963702
2,MLP,3,0.873672,"{'model__learning_rate_init': 0.001, 'model__h...",0.873595,0.492688,0.221365,0.323741,0.628697,0.375874,0.899640,0.367580,0.581315,0.414622,0.872487,9.980048


In [ ]:
all_results_df = pd.concat([results_df, mlp_results_df], ignore_index=True)

all_summary_df = (
    all_results_df.groupby("model", as_index=False)
    .agg(
        auc_mean=("auc", "mean"),
        auc_std=("auc", "std"),
        pr_auc_mean=("pr_auc", "mean"),
        logloss_mean=("logloss", "mean"),
        p11_mean=("precision@11", "mean"),
        r11_mean=("recall@11", "mean"),
        f11_mean=("f1@11", "mean"),
        hit11_mean=("hit@11", "mean"),
        pdk_mean=("precision@dynamic_k", "mean"),
        rdk_mean=("recall@dynamic_k", "mean"),
        fdk_mean=("f1@dynamic_k", "mean"),
        hitdk_mean=("hit@dynamic_k", "mean"),
        avg_pred_k=("avg_pred_k", "mean"),
    )
    .sort_values("auc_mean", ascending=False)
)

print("=== Summary including MLP ===")
all_summary_df

=== Summary including MLP ===


,model,auc_mean,auc_std,pr_auc_mean,logloss_mean,p11_mean,r11_mean,f11_mean,hit11_mean,pdk_mean,rdk_mean,fdk_mean,hitdk_mean,avg_pred_k
2,MLP,0.873970,0.000961,0.493455,0.222205,0.323938,0.623584,0.374355,0.89835,0.366607,0.578201,0.414086,0.8714,9.9772
0,LightGBM,0.873527,0.000464,0.493467,0.221731,0.323484,0.622837,0.373780,0.89785,0.366395,0.577597,0.413783,0.8712,9.9772
1,LogisticRegression_SAGA,0.858738,0.000614,0.457523,0.473376,0.314020,0.609376,0.363978,0.89560,0.352834,0.558211,0.399087,0.8646,9.9772


`Users`：用户数。
`Rows`：候选 `(user_id, product_id)` 总行数。
`Positive rate`：正样本占比（类别不平衡程度）。

`Filtered rule count`：Apriori 规则条数（阈值过滤后）。
`apriori_rule_hits mean`：命中规则的候选占比。

`inner_best_score_auc`：内层调参最优 AUC。
`auc / pr_auc`：越大越好。
`logloss`：越小越好。
`precision@11 / recall@11 / f1@11 / hit@11`：固定推荐 11 个时的效果。
`precision@dynamic_k / recall@dynamic_k / f1@dynamic_k / hit@dynamic_k`：动态推荐长度时的效果。
`avg_pred_k`：动态推荐平均长度（应接近用户平均篮子大小）。

`auc_mean`、`pr_auc_mean`、`f11_mean`、`fdk_mean` 比较模型。
LightGBM 在这些核心指标上都高于 Logistic Regression。

`mlp_results_df`含义与 `results_df` 一样，只是模型变成 MLP。
选型优先看：`pr_auc_mean` + `fdk_mean`，再 `logloss_mean`。